In [1]:
# Importamos las librerías básicas

import numpy as np
import pandas as pd

# modificamos la configuración para ver todas las columnas al mostrar dataframes
pd.set_option("display.max_columns", None)


In [2]:
# Carga de datasets

# Cargar dataset CSV (campañas de marketing)
bank_df = pd.read_csv( "../Data/DataRaw/bank-additional.csv", sep=",", index_col=0)  # usamos coma como separador

# Cargar dataset Excel (detalles de clientes con varias hojas)
customer_xlsx = pd.ExcelFile("../Data/DataRaw/customer-details.xlsx")
customer_2012 = pd.read_excel(customer_xlsx, sheet_name="2012", index_col=0)
customer_2013 = pd.read_excel(customer_xlsx, sheet_name="2013", index_col=0)
customer_2014 = pd.read_excel(customer_xlsx, sheet_name="2014", index_col=0)

# Utilizamos index_col=0 para indicar que la primera columna se use como índice

In [3]:
bank_df.sample(5)

,age,job,marital,education,default,housing,loan,contact,duration,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y,date,latitude,longitude,id_
38257,NaN,management,MARRIED,university.degree,0.0,0.0,0.0,cellular,411,2,999,0,NONEXISTENT,-3.4,"92,431","-26,9","0,742","5017,5",yes,19-mayo-2016,43.806,-69.073,164343f1-8c9c-44d7-8046-c0393d6ad55d
6411,25.0,technician,MARRIED,high.school,0.0,1.0,0.0,telephone,153,3,999,0,NONEXISTENT,1.1,"93,994","-36,4","4,857",5191,no,7-octubre-2017,34.008,-112.277,7ebe82ad-87bb-4695-92a9-c59b39376a7d
21452,31.0,management,MARRIED,university.degree,0.0,1.0,1.0,cellular,199,1,999,0,NONEXISTENT,1.4,"93,444","-36,1","4,963","5228,1",no,20-septiembre-2017,42.480,-72.959,debded44-a01d-44f1-ad03-bfb4eebbede8
20388,NaN,technician,MARRIED,high.school,0.0,1.0,0.0,cellular,237,1,999,0,NONEXISTENT,1.4,"93,444","-36,1","4,966","5228,1",no,18-marzo-2017,40.732,-75.137,744e0b74-d287-498b-bf57-ac6d6874d831
28907,41.0,management,MARRIED,university.degree,0.0,1.0,1.0,cellular,249,1,999,0,NONEXISTENT,-1.8,"93,075","-47,1","1,405","5099,1",no,27-mayo-2019,31.352,-116.024,0f1fd154-15da-4611-b1e0-7cd5b493ec45


Antes de realizar las transformaciones vamos a hacer una copia de nuestro set de datos para trabajar con ella. La primera transformación va a ser pasar la columna age de float a int ya que los valores son edades de los clientes que deben ser enteros.

In [4]:
df_bank_copy = bank_df.copy()

In [5]:
# Aplicamos lambda para convertir a int, cuidando los NaN
df_bank_copy['age'] = df_bank_copy['age'].apply(lambda x: int(x) if pd.notnull(x) else pd.NA)

# Convertimos a tipo entero de pandas (Int64) para mantener los NaN
df_bank_copy['age'] = df_bank_copy['age'].astype('Int64')

# Comprobamos
print("-> age dtype:", df_bank_copy['age'].dtype)


-> age dtype: Int64


Vamos a normalizar las tres columnas booleanas (default, housing, loan) usando map.
Haremos el reemplazo 0 → "no" y 1 → "yes", para tener la misma nomenclatura que la columna y.

In [6]:
# Normalizamos columnas booleanas con map (0 -> 'no', 1 -> 'yes')

bool_cols = ['default', 'housing', 'loan']

for col in bool_cols:
    df_bank_copy[col] = df_bank_copy[col].map({0: 'no', 1: 'yes'})
    print(f"-> {col}: valores únicos tras normalización:", df_bank_copy[col].unique())
    # mostramos los valores únicos de cada columna para comprobar que solo 
    # quedan "no" y "yes" (y NaN si existieran).


-> default: valores únicos tras normalización: ['no' nan 'yes']
-> housing: valores únicos tras normalización: ['no' 'yes' nan]
-> loan: valores únicos tras normalización: ['no' 'yes' nan]


In [7]:
df_bank_copy[bool_cols].sample(5)

,default,housing,loan
41076,no,yes,no
7017,no,no,no
28523,no,yes,no
29355,NaN,yes,no
32818,no,no,no


Vamos a convertir las columnas:
- 'cons.price.idx'
- 'cons.conf.idx'
- 'euribor3m'
- 'nr.employed'

De tipo object -> float

aplicamos .str.replace(',', '.') para pasar las comas decimales a puntos.

In [8]:
# Convertimos columnas 'object' con comas decimales a float

conv_float = ['cons.price.idx', 'cons.conf.idx', 'euribor3m', 'nr.employed']

for col in conv_float:
    if df_bank_copy[col].dtype == 'object': #nos aseguramos que es tipo obj
        # Reemplazar comas por puntos y convertir a float
        df_bank_copy[col] = df_bank_copy[col].str.replace(',', '.', regex=False).astype(float)
        print(f"-> {col} convertido a float")
    else:
        print(f"-> {col} ya es {df_bank_copy[col].dtype}, no necesita conversión")
    
#Comprobamos que ahora son tipo float
display(df_bank_copy[conv_float].sample(5))
df_bank_copy[conv_float].dtypes


-> cons.price.idx convertido a float
-> cons.conf.idx convertido a float
-> euribor3m convertido a float
-> nr.employed convertido a float


,cons.price.idx,cons.conf.idx,euribor3m,nr.employed
38497,92.431,-26.9,NaN,5017.5
28391,93.075,-47.1,NaN,5099.1
17278,93.918,-42.7,4.962,5228.1
40703,94.199,-37.5,0.879,4963.6
17571,93.918,-42.7,4.962,5228.1


cons.price.idx    float64
cons.conf.idx     float64
euribor3m         float64
nr.employed       float64
dtype: object

Reconsideramos que la columna nr.employed debería ser int ya que no tiene mucho sentido tener un número decimal de empleados. 

In [9]:
# Conversión de nr.employed a entero
    
# Truncamos a int (manteniendo NaN donde corresponda)
df_bank_copy['nr.employed'] = df_bank_copy['nr.employed'].apply(lambda x: int(x) if pd.notnull(x) else pd.NA)
    
# Convertimos a tipo entero de pandas (Int64) para mantener los NaN
df_bank_copy['nr.employed'] = df_bank_copy['nr.employed'].astype('Int64')
    
print("-> 'nr.employed' convertido a entero Int64")
df_bank_copy['nr.employed'].sample(5)


-> 'nr.employed' convertido a entero Int64


6842     5191
20150    5228
22219    5228
11437    5228
37900    5017
Name: nr.employed, dtype: Int64

Convertimos columna date de object a datetime

In [10]:
df_bank_copy['date'].sample(5)

26013         24-mayo-2017
23204        29-enero-2015
9083         10-abril-2016
18612        27-julio-2015
6386     13-diciembre-2019
Name: date, dtype: object

Mapeamos manualmente los meses en español y los convertimos a números

In [17]:
meses = {
    "enero":"01","febrero":"02","marzo":"03","abril":"04",
    "mayo":"05","junio":"06","julio":"07","agosto":"08",
    "septiembre":"09","octubre":"10","noviembre":"11","diciembre":"12"
}

#Mapeamos de forma manual, cambiamos meses en letra por su correspondiente número de mes
for mes, num in meses.items():
    df_bank_copy['date'] = df_bank_copy['date'].str.replace(mes, num, regex=True)

#Convertimos a datetime
df_bank_copy['date'] = pd.to_datetime(df_bank_copy['date'], dayfirst=True)

#Hacemos un sample para comprobar
df_bank_copy['date'].sample(5)

39657   2018-07-14
27617   2015-12-04
38425   2015-09-12
15492   2019-03-08
2525    2019-07-14
Name: date, dtype: datetime64[ns]

Creamos las columnas contact_month y contact_year a partir de la columna date

In [18]:
df_bank_copy["contact_month"] = df_bank_copy["date"].dt.month_name()
df_bank_copy["contact_year"] = df_bank_copy["date"].dt.year.astype('Int64')

# Verificamos el resultado en algunas filas
df_bank_copy[["date", "contact_month", "contact_year"]].head()


,date,contact_month,contact_year
0,2019-08-02,August,2019
1,2016-09-14,September,2016
2,2019-02-15,February,2019
3,2015-11-29,November,2015
4,2017-01-29,January,2017


Guardamos en la carpeta de DataProcessed el archivo con el dataframe limpio y con las transformaciones

In [19]:
df_bank_copy.to_csv("../Data/DataProcessed/bank-additional-clean.csv", index=False)
# Utilizamos index=False para evitar que se guarde la columna del índice como una columna extra en el CSV.